In [1]:
import MDAnalysis as mda
from numpy import *
import os
from pylab import *
import MDAnalysis.analysis.distances
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import glob
#import umap
import scipy.stats
import sklearn
import sklearn.decomposition
import sklearn.preprocessing
import pandas as pd
import seaborn as sns
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

In [2]:
import os

########################################################
#############      FOR NOW EQPOINT IS 0   ##############
########################################################
EQPOINT=0

systemFolders = glob.glob("huNumbering/*t5a*/")

systemgros=[]
systemtprs=[]
systemtrjs=[]
for i in range(len(systemFolders)):
    systemgros.append(sorted(glob.glob(systemFolders[i]+"*.gro")))
    systemtprs.append(sorted(glob.glob(systemFolders[i]+"*.tpr")))
    systemtrjs.append(sorted(glob.glob(systemFolders[i]+"*.xtc")))


    
    
threeColor=["#FE6100","#332288","#882255"]
colourScheme= threeColor
system_names = ["rhT5A","T5A","T5AR332P"]
systems=[]
for i in range(len(systemgros)):
    sub=[]
    for j in range(len(systemgros[i])):
        # When using TPRs, residues are indexed from 1; so we need to add in the first residue, 1 - 1 + first resid=first resid
        #firstres = mda.Universe(systemgros[i][j]).residues.resids[0]-1
        tu = mda.Universe(systemgros[i][j],systemtrjs[i][j])
        #tu.residues.resids +=firstres
                          
        sub.append(tu)
        
    systems.append(sub)


bodys=[]
bodystrings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and not ("+combinedLoopString+")"))
        sub2.append("protein and not ("+combinedLoopString+")")
        
    bodys.append(sub)
    bodystrings.append(sub2)
    
    
v1s=[]
v1strings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []

    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+combinedLoopString))
        sub2.append("protein and "+combinedLoopString)

        
    v1s.append(sub)        
    v1strings.append(sub2)

    
v2s=[]
v2strings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 380:405"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []

    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+combinedLoopString))
        sub2.append("protein and "+combinedLoopString)

        
    v2s.append(sub)        
    v2strings.append(sub2)
                   

C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\topology\base.py:203: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  residx = np.zeros_like(criteria[0], dtype=np.int)
C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\core\selection.py:640: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: h

In [3]:
def BestTwoD(array1d):
    #The number of dimensions (excluding nearest neighbours and self)
    N = int(1.5 + .5*np.sqrt(1+(8*len(array1d))))

    #Construct a 2D version of the contact map (it excludes nearesty neighbours and self, but there are still NxN slots)
    twoD=zeros((N,N))            
    j=2
    k=0
    for i in range(len(array1d)):
        if abs(j-k)>1:
            twoD[j][k] = array1d[i]
            twoD[k][j] = array1d[i]
        j+=1
        if j == len(twoD):
            k+=1
            j = k+2
            if k==len(twoD)-2 and j >len(twoD)-2 :
                break
    return twoD

In [4]:
allcontacts_ts=[]
for i in range(len(system_names)):
    allcontacts_ts.append(load("allcontacts_ts_"+system_names[i]+".npy"))

In [5]:
allcontacts_ts_2d = []
for sys in range(len(allcontacts_ts)):
    sub1=[]
    for i in range(len(allcontacts_ts[sys])):
        sub2=[]
        for j in range(len(allcontacts_ts[sys][i])):
            sub2.append(BestTwoD(allcontacts_ts[sys][i][j]))
        sub1.append(sub2)
    allcontacts_ts_2d.append(sub1)

for i in range(len(allcontacts_ts_2d)):
    save("allcontacts_ts_2d_"+system_names[i]+".npy",allcontacts_ts_2d[i])       

In [6]:
allcontacts_ts_2d=[]

for i in range(len(systems)):
    allcontacts_ts_2d.append(load("allcontacts_ts_2d_"+system_names[i]+".npy"))

In [7]:
for i in range(len(allcontacts_ts_2d)):
    allcontacts_ts_2d[i] = array(allcontacts_ts_2d[i])

MemoryError: Unable to allocate 11.2 GiB for an array with shape (9, 4001, 204, 204) and data type float64

In [ ]:
avg_v1_v2_contact_ts=[]
for i in range(len(allcontacts_ts_2d)):
    avg_v1_v2_contact_ts.append(mean(mean(allcontacts_ts_2d[i][:,:,324-290:436-290,380-290:405-290],axis = 2),axis = 2))
    
avg_v1_v3_contact_ts=[]
for i in range(len(allcontacts_ts_2d)):
    avg_v1_v3_contact_ts.append(mean(mean(allcontacts_ts_2d[i][:,:,324-290:436-290,415-290:430-290],axis = 2),axis = 2))
    
    
avg_v2_v3_contact_ts=[]
for i in range(len(allcontacts_ts_2d)):
    avg_v2_v3_contact_ts.append(mean(mean(allcontacts_ts_2d[i][:,:,380-290:405-290,415-290:430-290],axis = 2),axis = 2))

In [ ]:
imshow(allcontacts_ts_2d[0][0,0,324-290:350-290,380-290:405-290])

In [ ]:
#for i in range(len(avg_v1_v2_contact_ts)):
#    for j in range(len(avg_v1_v2_contact_ts[i])):
#        plot(avg_v1_v2_contact_ts[i][j],color = colourScheme[i],alpha = 0.2)
        
        
binrange = arange(-0.005,0.075,0.003)
figure(figsize = (0,0))

avg_v1_v2_contact_ts_hists = []
for i in range(len(avg_v1_v2_contact_ts)):
    sub=[]
    for j in range(len(avg_v1_v2_contact_ts[i])):
        a=hist(avg_v1_v2_contact_ts[i][j],color = colourScheme[i],alpha = 0.2,bins = binrange,density = True)
        sub.append(a[0])
    avg_v1_v2_contact_ts_hists.append(sub)
show()

figure(figsize = (16,10))

avg_hists=[]
sem_hists=[]
for i in range(len(avg_v1_v2_contact_ts_hists)):
    avg_hists.append(mean(avg_v1_v2_contact_ts_hists[i],axis = 0))
    sem_hists.append(scipy.stats.sem(avg_v1_v2_contact_ts_hists[i],axis = 0))
    plot(binrange[:-1],avg_hists[i],color = colourScheme[i])
    fill_between(binrange[:-1],avg_hists[i] - sem_hists[i], avg_hists[i] + sem_hists[i],color = colourScheme[i],alpha = 0.3)
    
xlabel("Mean V1-V2 Contact ($\AA$)",fontsize = 25)
ylabel("Probability Density",fontsize = 25)
title("V1-V2 Contact",fontsize = 25)
xticks(fontsize = 25)
yticks(fontsize = 25)

legend(fontsize = 25)

In [ ]:
#for i in range(len(avg_v1_v3_contact_ts)):
#    for j in range(len(avg_v1_v3_contact_ts[i])):
#        plot(avg_v1_v3_contact_ts[i][j],color = colourScheme[i],alpha = 0.2)
        
        
binrange = arange(-0.005,0.05,0.005)
figure(figsize = (0,0))

avg_v1_v3_contact_ts_hists = []
for i in range(len(avg_v1_v3_contact_ts)):
    sub=[]
    for j in range(len(avg_v1_v3_contact_ts[i])):
        a=hist(avg_v1_v3_contact_ts[i][j],color = colourScheme[i],alpha = 0.2,bins = binrange,density = True)
        sub.append(a[0])
    avg_v1_v3_contact_ts_hists.append(sub)
show()
   
figure(figsize = (16,10))
avg_hists=[]
sem_hists=[]
for i in range(len(avg_v1_v3_contact_ts_hists)):
    avg_hists.append(mean(avg_v1_v3_contact_ts_hists[i],axis = 0))
    sem_hists.append(scipy.stats.sem(avg_v1_v3_contact_ts_hists[i],axis = 0))
    plot(binrange[:-1],avg_hists[i],color = colourScheme[i])
    fill_between(binrange[:-1],avg_hists[i] - sem_hists[i], avg_hists[i] + sem_hists[i],color = colourScheme[i],alpha = 0.3)
    
xlabel("Mean V1-V3 Contact ($\AA$)",fontsize = 25)
ylabel("Probability Density",fontsize = 25)
title("V1-V3 Contact",fontsize = 25)
xticks(fontsize = 25)
yticks(fontsize = 25)

legend(fontsize = 25)

In [ ]:
#for i in range(len(avg_v2_v3_contact_ts)):
#    for j in range(len(avg_v2_v3_contact_ts[i])):
#        plot(avg_v2_v3_contact_ts[i][j],color = colourScheme[i],alpha = 0.2)
        
        
binrange = arange(-0.005,0.075,0.003)
figure(figsize = (0,0))
avg_v2_v3_contact_ts_hists = []
for i in range(len(avg_v2_v3_contact_ts)):
    sub=[]
    for j in range(len(avg_v2_v3_contact_ts[i])):
        a=hist(avg_v2_v3_contact_ts[i][j],color = colourScheme[i],alpha = 0.2,bins = binrange,density = True)
        sub.append(a[0])
    avg_v2_v3_contact_ts_hists.append(sub)
show()
    
    
figure(figsize = (16,10))
avg_hists=[]
sem_hists=[]
for i in range(len(avg_v2_v3_contact_ts_hists)):
    avg_hists.append(mean(avg_v2_v3_contact_ts_hists[i],axis = 0))
    sem_hists.append(scipy.stats.sem(avg_v2_v3_contact_ts_hists[i],axis = 0))
    plot(binrange[:-1],avg_hists[i],color = colourScheme[i])
    fill_between(binrange[:-1],avg_hists[i] - sem_hists[i], avg_hists[i] + sem_hists[i],color = colourScheme[i],alpha = 0.3)
    
xlabel("Mean V2-V3 Contact ($\AA$)",fontsize = 25)
ylabel("Probability Density",fontsize = 25)
title("V2-V3 Contact",fontsize = 25)
xticks(fontsize = 25)
yticks(fontsize = 25)

legend(fontsize = 25)

In [ ]:
# OK overall I'd say slight difference in V1-V2 conbtact between mutant and wild type.
# I'd also say no difference betweeb mutant and wild type for V1-V3 and V2-V3 contact.
# Given how extremely identical this V2-v3 contact is, it makes me think the differences we see in V1-V2 are actually real.

# What else?